# Мультиколлинеарность

In [ ]:
import pandas as pd
import numpy as np

from statsmodels.formula.api import ols
from statsmodels.graphics.api import plot_corr # визуализация корреляционной матрицы
from statsmodels.iolib.summary2 import summary_params # вывод результатов тестирования
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF # VIF

from scipy.stats import f # F-распределение и критические значения

import matplotlib.pyplot as plt
import seaborn as sns

# Не показывать Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)

<div style="background-color:Bisque; color:DarkBlue; padding:30px;">

<i><b><span style="color: purple">Мультиколлинеарность</span></b></i><br>

Матрицы регрессионного дизайна

$$
\begin{aligned}
\underset{n\times1}{y}&=\begin{pmatrix} y_1 \\ \vdots \\ y_n\end{pmatrix} &
\underset{n\times(k+1)}{X}&=\begin{pmatrix}
1 & x_{11} & \cdots & x_{1k} \\
1 & x_{21} & \cdots & x_{2k} \\
\vdots & \vdots & \ddots & \vdots \\
1 & x_{n1} & \cdots & x_{nk}\end{pmatrix}
\end{aligned}
$$

__"Чистая" мультиколлинеарность:__ $\det(X^\top X)=0$. В этом случае регрессия не подгоняется OLS-методом

__Мультиколлинеарность:__ $\det(X^\top X)\approx0$. Это приводит к "большим" стандартным ошибкам коэффициентов и, как следствие, к "маленьким" t-статистикам и к незначимости коэффициента

__Причина мультиколлинеарности__: высокая степень зависимости между регрессорами. В частности, сильная корреляция между регрессорами

__Идентификация мультиколлинеарности:__
* корреляционная матрица для регрессоров
* variance inflation factor $VIF=\frac{1}{1-R^2_j}$

**<span style="color:purple">Вывод:</span>** незначимость коэффициента при регрессоре может быть вызвана сильной корреляцией этой переменной с остальными

__Два практических кейса:__
* квадрат регрессора
* структурные сдвиги

</div>

Для набора данных `sleep75` рассмотрим регрессию

$$
sleep\;\text{на}\;totwrk,age,age^2,smsa,male,south,marr,union
$$

Подгоним модель и посмотрим значимость коэффициентов

In [ ]:
# импорт данных
df = pd.read_csv('../datasets/sleep75.csv')

## Спецификация и подгонка

In [ ]:
# спецификация модели через формулу
mod = ols(formula='sleep~1+totwrk+age+I(age ** 2)+smsa+male+south+marr+union', data=df)
# подгонка модели с неробастной оценкой ковариационной матрицы
res = mod.fit()

## Результаты неробастных t-тестов

In [ ]:
# уровень значимости
sign_level = 0.05

In [ ]:
# проверим значимость коэффициентов, используя P-значения
summary_params(res, alpha=sign_level).round(3)

**ВЫВОД**: на уровне значимости 5% (неробастные стандартные ошибки) значимы коэффициенты при регрессорах `totwrk`, `male`, `south`; незначимы коэффициенты при `age`, `I(age ** 2)`, `smsa`, `marr`, `union`.

## Корреляция регрессоров

In [ ]:
# матрица регрессионного дизайна X (первый столбец состоит только из единиц): mod.exog
# для наглядности представим в виде pandas DataFrame
X = pd.DataFrame(mod.exog, columns=mod.exog_names)
X

In [ ]:
# корреляционная матрица для регрессоров
# отметим, что нужно исключить первый столбец из единиц
corr_matrix = X.drop(columns='Intercept').corr()
corr_matrix.round(3)

In [ ]:
# визуализация корреляций
plot_corr(corr_matrix, xnames=mod.exog_names[1:], normcolor=True)
plt.show()

**ВЫВОД**: корреляция между `age` и `I(age ** 2)` равна 0.992, т.е. близка к единице. Остальные корреляции между регрессорами по модулю не превышают 0.38.

## VIF

In [ ]:
# имена экзогенных переменных
mod.exog_names

In [ ]:
# напечатаем VIF, используя индексы столбцов матрицы X
VIF(exog=X, exog_idx=1), VIF(exog=X, exog_idx=2), VIF(exog=X, exog_idx=3), VIF(exog=X, exog_idx=4)

In [ ]:
# напечатаем все VIF
for i in np.arange(1, len(mod.exog_names)):
    print(f'VIF({mod.exog_names[i]})={round(VIF(exog=X, exog_idx=i),3)}')

**ВЫВОД**: большой VIF у двух регрессоров: 66.010 у `age` и 66.168 у `I(age ** 2)`; у остальных регрессоров VIF не больше 1.22. Это значит, что дисперсия оценки коэффициента при `age` и при `I(age ** 2)` примерно в 66 раз больше, чем была бы при некоррелированных регрессорах. Принятый ориентир (не строгое правило): VIF > 10 указывает на мультиколлинеарность.

## Другие способы визуализации корреляционной матрицы

Ниже — другие способы показать ту же корреляционную матрицу.

С использованием метода [`heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html#seaborn.heatmap) из библиотеки [`seaborn`](https://seaborn.pydata.org/)

In [ ]:
# визуализация корреляций с помощью seaborn
sns.heatmap(corr_matrix, vmax=1, vmin=-1, annot=True)
plt.show()

Средствами `pandas`

In [ ]:
# визуализация корреляций с помощью pandas
corr_matrix.style.background_gradient(cmap='coolwarm', vmax=1, vmin=-1).format(precision=2)

Используя `matshow` из `matplotlib`

In [ ]:
# визуализация корреляций с помощью matplotlib
plt.matshow(corr_matrix)
plt.colorbar()
plt.show()

## F-тест: совместная значимость возраста

Тестируется значимость влияния возраста, т.е. гипотеза

$$
H_0:\beta_{age}=\beta_{age^2}=0
$$

Тест неробастный: используем OLS-оценку ковариационной матрицы

In [ ]:
# спецификация тестируемой гипотезы
F_test = res.f_test('age=0, I(age ** 2)=0')
print(F_test)

In [ ]:
# тестовая статистика и её P-значение
F_test.statistic, F_test.pvalue

In [ ]:
# степени свободы для F-распределения
F_test.df_num, F_test.df_denom

In [ ]:
# критические значения F-распределения на двух уровнях значимости
for sign_level in [0.05, 0.10]:
    F_cr = f.isf(q=sign_level, dfn=F_test.df_num, dfd=F_test.df_denom)
    print(f'sign_level={sign_level}: F_cr={F_cr:.3f}')

**ВЫВОД**: на уровне значимости 5% $F = 2.489 < F_{cr} = 3.009$, гипотеза $H_0$ не отвергается. Противоречия с t-тестами нет: коэффициенты при `age` и `I(age ** 2)` незначимы по отдельности (P-значения 0.531 и 0.376), и гипотеза об их совместной незначимости тоже не отвергается.

На уровне значимости 10% $F = 2.489 > F_{cr} = 2.310$, гипотеза $H_0$ отвергается, хотя коэффициенты при `age` и `I(age ** 2)` по отдельности незначимы и на этом уровне. На первый взгляд противоречие; оно объясняется мультиколлинеарностью.

**ВЫВОД**: мультиколлинеарность между `age` и `I(age ** 2)` объясняет, почему коэффициенты при этих регрессорах по отдельности незначимы: их стандартные ошибки раздуты. По отдельным t-тестам нельзя заключить, что возраст не влияет на `sleep`: F-тест отвергает совместную незначимость `age` и `I(age ** 2)` на уровне значимости 10% (P-значение 0.084), хотя на уровне 5% не отвергает. Подробно F-тест разобран в `f-test-joint.ipynb`.